In [1]:
from pathlib import Path
from typing import Optional

import pandas as pd, numpy as np
from utils import *

In [2]:
df_simule = pd.DataFrame(pd.read_csv(r"../../datas/bronze/dataset_brut.csv"))
df_iot = pd.DataFrame(pd.read_csv(r"../../datas/silver/dataset_iot.csv"))
df_plc = pd.DataFrame(pd.read_csv(r"../../datas/silver/dataset_plc.csv"))

In [ ]:
df_simule.head()

,timestamp,machine_id,type_machine,secteur,type_metal,vitesse_rotation_nominal,courant_moteur_nominal,pression_hydraulique_nominal,statut_nominal,temp_base_moteur,...,iot_vitesse_rotation,iot_courant_moteur,iot_pression_hydraulique,iot_temperature,iot_vibration_rms,iot_vibration_peak,iot_charge_moteur,age_jours,age_virtuel_jours,RUL_jours
0,2024-08-01 00:00:00,MCH-001,CNC_5_Axes,Aéro,Aluminium,6000,20.0,0.0,Production,40.0,...,6000.0,19.808858,0.000000,48.191631,0.914319,1.245143,45.0,1200.000000,840.000000,0.793750
1,2024-08-01 00:00:30,MCH-001,CNC_5_Axes,Aéro,Aluminium,6000,20.0,0.0,Production,40.0,...,6000.0,20.070450,0.000000,48.062069,0.874214,1.380316,45.0,1200.000347,840.000347,0.793403
2,2024-08-01 00:01:00,MCH-001,CNC_5_Axes,Aéro,Aluminium,6000,20.0,0.0,Production,40.0,...,6000.0,20.052242,0.000000,48.706761,0.858751,1.230252,45.0,1200.000694,840.000694,0.793056
3,2024-08-01 00:01:30,MCH-001,CNC_5_Axes,Aéro,Aluminium,6000,20.0,0.0,Production,40.0,...,6000.0,20.036585,0.000000,48.484258,0.849300,1.275070,45.0,1200.001042,840.001042,0.792708
4,2024-08-01 00:02:00,MCH-001,CNC_5_Axes,Aéro,Aluminium,6000,20.0,0.0,Production,40.0,...,6000.0,19.829501,0.009466,48.398311,0.861642,1.243091,45.0,1200.001389,840.001389,0.792361


In [ ]:
df_iot.head()

,timestamp,machine_id,iot_vitesse_rotation,iot_courant_moteur,iot_pression_hydraulique,iot_temperature,iot_vibration_peak,iot_charge_moteur
0,2024-08-01 00:00:00,MCH-001,6000.0,19.808858,0.000000,48.191631,1.245143,45.0
1,2024-08-01 00:00:00,MCH-006,0.0,54.883773,179.429525,61.537411,3.042907,70.0
2,2024-08-01 00:00:00,MCH-004,6200.0,19.072288,0.087864,44.327032,1.162757,45.0
3,2024-08-01 00:00:00,MCH-007,4800.0,15.925603,0.000000,38.907229,0.498840,45.0
4,2024-08-01 00:00:00,MCH-002,4500.0,22.086468,0.001289,41.105098,1.404059,45.0


In [ ]:
df_plc.head()

,machine_id,timestamp,iot_statut_machine,type_metal
0,MCH-001,2024-08-01 00:00:00,Production,Aluminium
1,MCH-006,2024-08-01 00:00:00,Production,Acier
2,MCH-004,2024-08-01 00:00:00,Production,Aluminium
3,MCH-007,2024-08-01 00:00:00,Production,Aluminium
4,MCH-002,2024-08-01 00:00:00,Production,Aluminium


In [ ]:
df_type_machine = df_simule[[
    "type_machine"
]].drop_duplicates(subset=[
        "type_machine",
    ]
)
df_type_machine["id"] = df_type_machine["type_machine"].astype("category").cat.codes + 1
df_type_machine.head(20)

,type_machine,id
0,CNC_5_Axes,2
4204800,CNC_3_Axes,1
8409600,Robot_Laser,4
12614400,Presse_Hydr,3


In [ ]:
df_type_metal=df_simule[[
    "type_metal"
]].drop_duplicates(subset=[
        "type_metal",
    ]
)
df_type_metal["id"] = df_type_metal["type_metal"].astype("category").cat.codes + 1
df_type_metal.head(20)

,type_metal,id
0,Aluminium,2
480,Titane,4
5760,Aucun,3
12614400,Acier,1


In [ ]:
df_plc = df_plc.merge(
    df_type_metal[["type_metal", "id"]].rename(columns={"id": "id_type_metal"}),
    on="type_metal",
    how="left"
).drop(columns="type_metal")
df_plc.head()

,machine_id,timestamp,iot_statut_machine,id_type_metal
0,MCH-001,2024-08-01 00:00:00,Production,2
1,MCH-006,2024-08-01 00:00:00,Production,1
2,MCH-004,2024-08-01 00:00:00,Production,2
3,MCH-007,2024-08-01 00:00:00,Production,2
4,MCH-002,2024-08-01 00:00:00,Production,2


In [ ]:
df_gmao_label = df_simule[[
    "label_gmao"
]].drop_duplicates(subset=[
        "label_gmao",
    ]
)
df_gmao_label["id"] = df_gmao_label["label_gmao"].astype("category").cat.codes + 1
df_gmao_label.head()

,label_gmao,id
0,Sain,20
1338,Alerte_P4,4
2287,Maintenance_Correctif_P4,13
16131,Alerte_P10,2
16133,Maintenance_Correctif_P10,11


In [ ]:
df_production_status = df_simule[[
    "iot_statut_machine"
]].drop_duplicates(subset=[
        "iot_statut_machine",
    ]
)
df_production_status["id"] = df_production_status["iot_statut_machine"].astype("category").cat.codes + 1
df_production_status.head()

,iot_statut_machine,id
0,Production,4
2286,Panne,3
2287,Maintenance,2
5760,Arrêt Opérateur,1


In [ ]:
df_plc.head()

,machine_id,timestamp,iot_statut_machine,id_type_metal
0,MCH-001,2024-08-01 00:00:00,Production,2
1,MCH-006,2024-08-01 00:00:00,Production,1
2,MCH-004,2024-08-01 00:00:00,Production,2
3,MCH-007,2024-08-01 00:00:00,Production,2
4,MCH-002,2024-08-01 00:00:00,Production,2


In [ ]:
df_plc = df_plc.merge(
    df_production_status[["iot_statut_machine", "id"]].rename(columns={"id": "id_status_production"}),
    on="iot_statut_machine",
    how="left"
).drop(columns="iot_statut_machine")
df_plc.head()

,machine_id,timestamp,id_type_metal,id_status_production
0,MCH-001,2024-08-01 00:00:00,2,4
1,MCH-006,2024-08-01 00:00:00,1,4
2,MCH-004,2024-08-01 00:00:00,2,4
3,MCH-007,2024-08-01 00:00:00,2,4
4,MCH-002,2024-08-01 00:00:00,2,4


In [ ]:
df_nominale = df_simule[["timestamp","secteur", "machine_id", "vitesse_rotation_nominal", "courant_moteur_nominal", "pression_hydraulique_nominal", "statut_nominal", "temp_base_moteur"]]

In [ ]:
from utils import record_future_send_in_jsonl

folder_path = Path("../../datas/gold")
folder_path.mkdir(parents=True, exist_ok=True)
df_type_machine.to_csv(
    name_csv_file(
        folder_path=folder_path,
        filename="type_machine",
        extension=".csv",
        type_dst="postgres"
    ), index=False, encoding='utf-8'
)
df_type_metal.to_csv(
    name_csv_file(
        folder_path=folder_path,
        filename="type_metal",
        extension=".csv",
        type_dst="postgres"
    ), index=False, encoding='utf-8'
)
df_production_status.to_csv(
    name_csv_file(
        folder_path=folder_path,
        filename="production_status",
        extension=".csv",
        type_dst="postgres"
    ), index=False, encoding='utf-8'
)
df_nominale.to_csv(
    name_csv_file(
        folder_path=folder_path,
        filename="nominale_values",
        extension=".csv",
        type_dst="postgres"
    ), index=False, encoding='utf-8'
)

In [ ]:
df_maintenance = remove_rows_containing_string_in_column(
    df=df_simule,
    column_name="label_gmao",
    string_to_remove="Sain"
)

In [ ]:
df_alerte_splited, df_maintenance_splited = split_dataframe_by_prefix(df_maintenance, "label_gmao","Alerte")
df_maintenance_unique = create_table_with_id_and_unique_label(df_maintenance_splited,"label_gmao")
df_alerte_unique = create_table_with_id_and_unique_label(df_alerte_splited,"label_gmao")

In [ ]:
folder_path = Path("../../datas/gold")
df_maintenance_unique.to_csv(
    name_csv_file(
        folder_path=folder_path,
        filename="maintenance",
        extension=".csv",
        type_dst="postgres"
    ), index=False, encoding='utf-8'
)
df_alerte_unique.to_csv(
    name_csv_file(
        folder_path=folder_path,
        filename="alerte",
        extension=".csv",
        type_dst="postgres"
    ), index=False, encoding='utf-8'
)

In [3]:
df_age = build_machine_age_dataframe(df_simule)

In [ ]:
df_age.head()

In [7]:
df_age.to_csv(
    name_csv_file(
        folder_path=folder_path,
        filename="age_machine",
        extension=".csv",
        type_dst="postgres"
    ), index=False, encoding='utf-8'
)

In [ ]:
df_machine = build_machine_dataframe(df_simule, df_type_machine)
df_machine.to_csv(
    name_csv_file(
        folder_path=folder_path,
        filename="machine",
        extension=".csv",
        type_dst="postgres"
    ), index=False, encoding='utf-8'
)

df_iot = (
    df_iot.sort_values("timestamp")
      .reset_index(drop=True)
)
df_plc = (
    df_plc.sort_values("timestamp")
      .reset_index(drop=True)
)
record_future_send_in_jsonl(df_iot, df_plc, output_path=r"../../datas/gold/mqtt_iot_plc_send.jsonl")